<h3>Single image processing (with napari visualization)</h3>

### Import dependencies and select GPU device
Load utility functions and analysis libraries required for segmentation, quantification, and visualization.

In [ ]:
from utils import (
    extract_scaling_metadata,
    predict_nuclei_labels, 
    simulate_cytoplasm, 
    segment_organoids_from_cp_labels, 
    map_df_column_to_labels
)
from data_analysis_utils import extract_organoid_stats_and_merge
from io_utils import (
    list_images, 
    ensure_output_dir, 
    load_precomputed_results_if_available
)
from pathlib import Path
import tifffile
import czifile
import napari
import numpy as np
from skimage.measure import regionprops_table
import pandas as pd
import pyclesperanto_prototype as cle
import plotly.express as px

cle.select_device("RTX")

### Configure experiment inputs
Define data location and per channel info used in the workflow.

Input needed from researcher: update RAW_DATA_DIRECTORY, NUCLEI_CHANNEL, MIN_MAX_NUCLEI_VOLUME, channel indices, marker definitions, and index for the image to explore from your input directory.

In [ ]:
# Copy the path where your .czi files are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = "./raw_data/20240607 PGE2MSN WT_organoids"
#RAW_DATA_DIRECTORY = "./raw_data/20240703 PGE2MSN Tum_organoids"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# Minimum and maximum nuclei label volume to use for filtering predicted nuclei labels (min, max) i.e (250, 4000)
# If set to None, no filtering is applied and all predicted labels are returned
MIN_MAX_NUCLEI_VOLUME = (250, 4000)

# Cytoplasm thickness in voxels (starting from the nucleus)
# If padding is applied, the thickness will be applied to the padded nucleus
CYTOPLASM_THICKNESS = 2

# Padding to keep a safe distance between nuclei and cytoplasm borders
# It expands the nucleus by the padding value in all directions
NUCLEI_PADDING = 0

# List of markers used in the experiment, each with its corresponding channel index
MARKERS = [("UEA-1", 0), ("YAP", 1), ("Nuclei", 2), ("BCAT", 3)]

# Mark the position of the .czi iamge you want to open in your raw data folder (first = 0, second = 1, third = 2)
CZI_IMAGE_INDEX = 1

### Load selected image and parse identifiers
Read the chosen `.czi` file, squeeze dimensions, and extract experiment metadata from filename fields.

Input needed from researcher: ensure your filename format supports experiment, mouse, treatment, and replica parsing.

In [ ]:
directory_path = Path(RAW_DATA_DIRECTORY)
input_folder_id = directory_path.stem
images = list_images(directory_path, "czi")

image = images[CZI_IMAGE_INDEX]

# Read path storing raw image and extract filename
file_path = Path(image)
filename = file_path.stem

# Read the image file and remove singleton dimensions
img = czifile.imread(image)
img = img.squeeze()

# Extract experiment, mouse, treatment and replica ids
experiment_id = filename.split(" ")[0]
mouse_id = filename.split(" ")[1]
treatment_id = filename.split(" ")[2]
replica_id = filename.split(" ")[-1]


### Launch 3D napari viewer
Open the raw channels in napari for visual quality control before segmentation and quantification.

Input needed from researcher: verify channel names, colors, and orientation are correct.

In [ ]:
# Initialize Napari Viewer and display all input channels
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(img, 
                channel_axis=0,
                colormap=['white', 'magenta', 'cyan', 'yellow'],
                name=[tuple[0] for tuple in MARKERS]
                )

### Nuclei label prediction using CellposeSAM & cytoplasm simulation
This step loads existing nuclei labels if present, or predicts and saves new labels.
Then based on those nuclei labels it performs a morphological dilation operation to simulate the surrounding cytoplasm

In [ ]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_output_dir(RAW_DATA_DIRECTORY, input_folder_id, results_type="nuclei_labels")
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
# Extract x,y,z scaling from .czi file metadata
scaling_x_um, scaling_y_um, scaling_z_um = extract_scaling_metadata(file_path)

# Calculate anisotropy to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = scaling_z_um / np.mean([scaling_x_um , scaling_y_um])

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_results_if_available(nuclei_labels_dir, filename, results_type="nuclei_labels")

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {filename} ...loading")
    # If nuclei_labels are loaded from disk, filter by MIN_MAX_NUCLEI_VOLUME if specified
    if MIN_MAX_NUCLEI_VOLUME is not None:
        from utils import _keep_objects_in_size_range  # safe in notebook
        nuclei_labels = _keep_objects_in_size_range(nuclei_labels, MIN_MAX_NUCLEI_VOLUME)

    # Add the prediction to the napari viewer
    viewer.add_labels(nuclei_labels)
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(img, rescale_factor, NUCLEI_CHANNEL, MIN_MAX_NUCLEI_VOLUME, visualize=True)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{filename}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)

# Simulate cytoplasm based on input parameters
cytoplasm = simulate_cytoplasm(nuclei_labels, cytoplasm_thickness=CYTOPLASM_THICKNESS, nuclei_padding=NUCLEI_PADDING)
viewer.add_labels(cytoplasm)

# Segment organoids using Cellpose generated labels as seeds
organoid_labels = segment_organoids_from_cp_labels(nuclei_labels, cytoplasm_thickness=1)
organoid_mask = organoid_labels > 0
viewer.add_image(organoid_mask, opacity=0.5)

# Create a copy of the original_nuclei labels to avoid in place modifications
nuclei_labels_filtered = nuclei_labels.copy()

# Filter out any nuclei that sits outside the organoid (in x,y)
nuclei_labels_filtered[~organoid_mask] = 0

# Add the filtered nuclei labels to the viewer
viewer.add_labels(nuclei_labels_filtered)

### Compute YAP nuclei-to-cytoplasm ratio and map nuclei labels to organoid parents
Extract intensity and volume features for nuclei and simulated cytoplasm, then compute per-label ratios.
Nuclei not belonging to any organoid can be filtered out during data analysis since these are mapped to organoid_id == 0.

In [ ]:
# Automatically find the YAP channel index in MARKERS and extract the corresponding stack from img
yap_channel_index = next(i for i, marker in enumerate(MARKERS) if "YAP" in marker[0])

#Extract regionprops from filtered nuclei and generated-cytoplasm labels
props_nuclei = regionprops_table(label_image=nuclei_labels, intensity_image=img[yap_channel_index], properties=["label", "intensity_mean", "area"])
props_cytoplasm = regionprops_table(label_image=cytoplasm, intensity_image=img[yap_channel_index], properties=["label", "intensity_mean", "area"])


# Transform regionprops_table into a Dataframe to process it using Pandas
df_nuclei = pd.DataFrame(props_nuclei)
df_cytoplasm = pd.DataFrame(props_cytoplasm)

# Renaming columns
df_nuclei.rename(columns={'intensity_mean': 'intensity_mean_nuclei', 'area': 'volume_nuclei'}, inplace=True)
df_cytoplasm.rename(columns={'intensity_mean': 'intensity_mean_cyto', 'area': 'volume_cyto'}, inplace=True)

# Merge dataframes on label
merged_df = pd.merge(df_nuclei, df_cytoplasm, on='label')

# Calculate nuclei/cytoplasm ratio of mean intensity of YAP signal
merged_df['nuclei_cyto_ratio'] = merged_df['intensity_mean_nuclei'] / merged_df['intensity_mean_cyto']

# Extract organoid stats and map nuclei_labels to parent organoid_labels
# Nuclei labels not mapped to an organoid get an organoid_id mapping of 0 value
merged_df = extract_organoid_stats_and_merge(nuclei_labels, organoid_labels, merged_df)

# Add extracted ID info to the dataframe
merged_df.insert(0, 'experiment_id', experiment_id)
merged_df.insert(1, 'mouse_id', mouse_id)
merged_df.insert(2, 'treatment_id', treatment_id)
merged_df.insert(3, 'replica_id', replica_id)

merged_df.head(10)

### Query a specific label
Inspect per-label metrics for one nucleus to validate segmentation and intensity outputs.

Input needed from researcher: replace the label id with the nucleus of interest.

In [ ]:
# Explore particular label results (i.e. 541)
label = 541

merged_df[merged_df['label']==label]

### Inspect nuclei volume distribution
Review label volumes with a histogram to calibrate size-based filtering thresholds.

Input needed from researcher: inspect outliers and adjust `MIN_MAX_NUCLEI_VOLUME` if needed.

In [ ]:
# Explore nuclei predicted labels by Cellpose to make an informed decision on the size-based exclusion of the predicted labels
# Create histogram
fig = px.histogram(merged_df, x="volume_nuclei", nbins=300, labels={"volume_nuclei": "Volume"}, title="Histogram of volumes of Cellpose predicted labels")

# Show plot
fig.show()

### Visualize measurements on nuclei labels
Project quantitative outputs back onto the nuclei labels to quickly inspect spatial patterns.

Input needed from researcher: choose the dataframe column to display and the colormap that best highlights your biological contrast.

In [ ]:
# Mappings before filtering nuclei not belonging to any organoid
# map_df_column_to_labels(nuclei_labels, merged_df, value_column="volume_nuclei", colormap="viridis", visualize=True)
# map_df_column_to_labels(nuclei_labels, merged_df, value_column="nuclei_cyto_ratio", colormap="inferno", visualize=True)

# Mappings after filtering
map_df_column_to_labels(nuclei_labels_filtered, merged_df, value_column="volume_nuclei", colormap="viridis", visualize=True)
map_df_column_to_labels(nuclei_labels_filtered, merged_df, value_column="nuclei_cyto_ratio", colormap="inferno", visualize=True)

In [ ]:
map_df_column_to_labels(nuclei_labels, merged_df, value_column="organoid_id", colormap="viridis", visualize=True)